# CLEIDS-Edge — Notebook 02: Architecture Definition

Defines the CLEIDS-Edge hybrid CNN-LSTM architecture as a **model-building function parameterized by `input_dim`** (and `num_classes` for the multiclass head) — not a model with a hardcoded input shape. This is required because the four (soon five) datasets have different feature counts after per-dataset encoding (Notebook 01): NSL-KDD=122, CICIDS2017=78, UNSW-NB15=194, TON_IoT=76, IoT-23=pending. Notebook 03 will call `build_cleids_edge()` once per dataset.

**Executed locally, not in Colab** — no GPU is needed for architecture definition and dummy-batch sanity checks (only Notebook 03's actual training needs the GPU runtime). TensorFlow was installed locally for this session.

**No training on real data happens in this notebook** — only dummy random batches, to confirm the architecture is structurally sound across every known `input_dim` before Notebook 03 spends real compute on it.

## 1. Setup

In [ ]:
import os
import sys
import io
import contextlib
import json

import numpy as np
import tensorflow as tf

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU visible:", tf.config.list_physical_devices("GPU"))
print("(No GPU expected/needed here -- this notebook only defines the architecture "
      "and runs dummy-batch sanity checks. Notebook 03's actual training uses the Colab GPU runtime.)")

sys.path.insert(0, os.path.join(os.getcwd(), "src") if os.path.basename(os.getcwd()) != "src" else os.getcwd())
from models import build_cleids_edge

with open("data/processed/preprocessing_manifest.json") as f:
    prep_manifest = json.load(f)
INPUT_DIMS = {name: info["n_features"] for name, info in prep_manifest["datasets"].items()}
print("\nConfirmed input_dim per dataset (from Notebook 01's preprocessing_manifest.json):")
for name, dim in INPUT_DIMS.items():
    print(f"  {name}: {dim}")
print("  iot-23: pending (not yet available -- architecture must not hardcode any dataset's shape)")

## 2. Architecture definition

`build_cleids_edge(input_dim, num_classes=2, binary=True)` lives in `src/models.py` (not redefined here) so Notebook 03 can import it directly. Implements the specified design exactly:

Input `(input_dim, 1)` → Conv1D(64, k=3, ReLU) → BatchNorm → MaxPool(2) → Conv1D(128, k=3, ReLU) → BatchNorm → MaxPool(2) → Dropout(0.3) → LSTM(100) → Dropout(0.3) → Dense(64, ReLU) → output head. Adam(lr=0.001); binary head is sigmoid/binary-crossentropy, multiclass head is softmax/categorical-crossentropy. Metrics: accuracy, AUC, Precision, Recall.

**No deviations from the given spec** — layer sizes, dropout rates, LSTM units, and optimizer settings are exactly as specified. The only unspecified detail was Conv1D padding; Keras' default (`'valid'`, no padding) was kept rather than substituting `'same'`, since it was not overridden in the given spec and every known `input_dim` (76–194) comfortably survives two `valid`-padded conv+pool reductions without hitting a zero/negative sequence length (verified below, not assumed).

**Notable, non-obvious property**: total parameter count (123,857) is **identical across all four datasets** despite their different `input_dim`. This is because Conv1D/BatchNorm/Dense parameter counts depend only on the number of filters/units (fixed: 64, 128, 100, 64), not on sequence length — and the LSTM (`return_sequences=False`) only consumes the final hidden state, so its parameter count is likewise independent of sequence length. `input_dim` only changes the *intermediate activation shapes* (and therefore compute/FLOPs), not the parameter count or model file size.

In [ ]:
import inspect
print(inspect.getsource(build_cleids_edge))

## 3. Parameter count report + sanity checks (per dataset input_dim)

In [ ]:
results = {}

for name, input_dim in INPUT_DIMS.items():
    print("\n" + "=" * 70)
    print(f"{name} (input_dim={input_dim})")
    print("=" * 70)

    model = build_cleids_edge(input_dim, num_classes=2, binary=True)

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        model.summary()
    print(buf.getvalue())

    total_params = model.count_params()
    trainable_params = sum(int(tf.size(w)) for w in model.trainable_weights)
    non_trainable_params = sum(int(tf.size(w)) for w in model.non_trainable_weights)
    size_mb = total_params * 4 / (1024 ** 2)

    print(f"Total params: {total_params:,}")
    print(f"Trainable: {trainable_params:,}  Non-trainable: {non_trainable_params:,}")
    print(f"Estimated size (float32, params*4 bytes): {size_mb:.3f} MB")

    # Sanity check: dummy batch forward pass
    X_dummy = np.random.randn(32, input_dim, 1).astype(np.float32)
    y_pred = model.predict(X_dummy, verbose=0)
    shape_ok = y_pred.shape == (32, 1)
    nan_inf_ok = not (np.isnan(y_pred).any() or np.isinf(y_pred).any())
    print(f"[SANITY] output shape: {y_pred.shape} -- {'OK' if shape_ok else 'MISMATCH'}")
    print(f"[SANITY] NaN/Inf in output: {'NONE -- OK' if nan_inf_ok else 'FOUND -- FAIL'}")

    # Sanity check: model.fit() executes end-to-end for 1 epoch (dummy labels, no real training)
    y_dummy = np.random.randint(0, 2, size=(32, 1)).astype(np.float32)
    try:
        history = model.fit(X_dummy, y_dummy, epochs=1, batch_size=8, verbose=0)
        fit_ok = True
        fit_loss = history.history["loss"][-1]
    except Exception as e:
        fit_ok = False
        fit_loss = None
        print(f"[SANITY] model.fit() FAILED: {e}")
    print(f"[SANITY] model.fit() 1 epoch: {'OK (loss=' + str(fit_loss) + ')' if fit_ok else 'FAILED'}")

    results[name] = {
        "input_dim": input_dim, "total_params": total_params,
        "trainable_params": trainable_params, "non_trainable_params": non_trainable_params,
        "size_mb": size_mb, "shape_ok": shape_ok, "nan_inf_ok": nan_inf_ok, "fit_ok": fit_ok,
    }

## 4. Multi-class variant check

`binary=False, num_classes=15` (CICIDS2017's class count — the largest of the four), at `input_dim=78`.

In [ ]:
mc_model = build_cleids_edge(78, num_classes=15, binary=False)
X_dummy = np.random.randn(32, 78, 1).astype(np.float32)
y_pred = mc_model.predict(X_dummy, verbose=0)
mc_shape_ok = y_pred.shape == (32, 15)
mc_nan_inf_ok = not (np.isnan(y_pred).any() or np.isinf(y_pred).any())
mc_sums_to_one = np.allclose(y_pred.sum(axis=1), 1.0, atol=1e-4)
print(f"[SANITY] output shape: {y_pred.shape} -- {'OK' if mc_shape_ok else 'MISMATCH'}")
print(f"[SANITY] NaN/Inf in output: {'NONE -- OK' if mc_nan_inf_ok else 'FOUND -- FAIL'}")
print(f"[SANITY] softmax rows sum to 1: {'OK' if mc_sums_to_one else 'FAIL'}")

y_dummy_onehot = tf.keras.utils.to_categorical(np.random.randint(0, 15, size=(32,)), num_classes=15)
try:
    mc_model.fit(X_dummy, y_dummy_onehot, epochs=1, batch_size=8, verbose=0)
    mc_fit_ok = True
except Exception as e:
    mc_fit_ok = False
    print(f"[SANITY] model.fit() FAILED: {e}")
print(f"[SANITY] model.fit() 1 epoch: {'OK' if mc_fit_ok else 'FAILED'}")

## 5. Baseline architecture stubs (interfaces for Notebook 04)

Signatures + docstrings only, per `CLEIDS_PROJECT_BRIEF.md` §3 — full implementations happen in Notebook 04, not here. Defined in `src/models.py` alongside `build_cleids_edge`.

In [ ]:
import models as models_module

BASELINE_STUBS = [
    "build_random_forest", "build_svm", "build_standalone_cnn", "build_standalone_lstm",
    "build_nazir2024_hybrid", "build_altaie_hoomod2024", "build_wang2023_dlbilstm",
    "build_misrak_melaku2025",
]
for fn_name in BASELINE_STUBS:
    fn = getattr(models_module, fn_name)
    doc_first_line = fn.__doc__.strip().splitlines()[0]
    print(f"{fn_name}{inspect.signature(fn)} -- {doc_first_line}")

## 6. Module file + push

`build_cleids_edge` and the 8 baseline stubs live in `src/models.py` at the repo root (imported above, not redefined in this notebook) so Notebook 03/04 can `from src.models import build_cleids_edge` directly. The file is committed and pushed to GitHub alongside this notebook -- future sessions cloning the repo (Notebook 00, cell 1) get it automatically.

In [ ]:
print("src/models.py exists:", os.path.exists("src/models.py"))
# Executed locally (see markdown at top) -- notebook and src/models.py are committed/pushed
# via git directly from this machine, consistent with Notebooks 00/01.

## 7. Final summary

Paste this cell's output back for review before Notebook 03.

In [ ]:
print("=" * 70)
print("CLEIDS-Edge -- Notebook 02 Summary")
print("=" * 70)
for name, r in results.items():
    print(f"[{name}] input_dim={r['input_dim']:<4} total_params={r['total_params']:>10,} "
          f"trainable={r['trainable_params']:>10,} non_trainable={r['non_trainable_params']:>6,} "
          f"size={r['size_mb']:.3f}MB shape_ok={r['shape_ok']} nan_inf_ok={r['nan_inf_ok']} fit_ok={r['fit_ok']}")
print(f"\n[multiclass] input_dim=78 num_classes=15 shape_ok={mc_shape_ok} "
      f"nan_inf_ok={mc_nan_inf_ok} sums_to_one={mc_sums_to_one} fit_ok={mc_fit_ok}")

all_pass = all(r["shape_ok"] and r["nan_inf_ok"] and r["fit_ok"] for r in results.values()) and \
    mc_shape_ok and mc_nan_inf_ok and mc_fit_ok
print(f"\nALL SANITY CHECKS PASSED: {all_pass}")
print("\nNote: total parameter count is identical across all input_dim values (123,857) -- "
      "expected, since Conv1D/BatchNorm/LSTM/Dense parameter counts depend on filter/unit "
      "counts, not sequence length. input_dim only changes intermediate activation shapes.")
print("\niot-23: architecture is input_dim-agnostic by construction -- no changes needed "
      "once its feature count is known from Notebook 01's IoT-23 section.")
print("\nNext: Notebook 03 (Training CLEIDS-Edge) -- Colab GPU runtime required.")